# Lesson 04 - Mẫu Thiết Kế Sử Dụng Công Cụ

Trong bài học này, bạn sẽ học mẫu thiết kế **Sử Dụng Công Cụ** cho các đại lý AI sử dụng Microsoft Agent Framework (Python). Chúng ta sẽ đề cập đến:

- Định nghĩa các công cụ hàm với bộ trang trí `@tool` và các tham số có kiểu
- Cung cấp sơ đồ công cụ để mô hình biết mỗi công cụ làm gì
- Kiểm soát việc thực thi công cụ bằng `approval_mode`
- Trả về **đầu ra có cấu trúc** qua các mô hình Pydantic và `response_format`

Kịch bản là một **đại lý đặt chỗ du lịch** có thể tra cứu điểm đến, kiểm tra tình trạng, và lấy thông tin chuyến bay.


## Cài đặt


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Định nghĩa Công cụ với Bộ trang trí @tool

Bộ trang trí `@tool` biến một hàm Python thông thường thành một công cụ mà tác nhân có thể gọi.
Các điểm chính:

- **Chuỗi tài liệu** trở thành mô tả công cụ mà mô hình nhìn thấy.
- **Chú thích kiểu** (bao gồm `Annotated` với các mô tả) xác định lược đồ công cụ.
- `approval_mode` kiểm soát việc liệu người dùng có phải phê duyệt mỗi lần gọi trước khi nó thực thi hay không.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Tạo một Agent với Nhiều Công Cụ

Truyền cả ba công cụ cho client để mô hình có thể gọi bất kỳ công cụ nào nó cần để trả lời câu hỏi của người dùng.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Đầu Ra Có Cấu Trúc với Công Cụ

Bằng cách thiết lập `response_format` thành một mô hình Pydantic, đại lý bị bắt buộc phải trả về một đối tượng JSON kiểu rõ ràng thay vì văn bản tự do. Điều này hữu ích khi mã phía dưới cần tiêu thụ kết quả một cách có lập trình.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Mẫu phê duyệt công cụ

Tham số `approval_mode` trên `@tool` kiểm soát xem các lần gọi công cụ có yêu cầu sự phê duyệt của con người trước khi thực thi hay không:

| Chế độ | Hành vi |
|---|---|
| `"never_require"` | Công cụ chạy tự động — không cần xác nhận từ người dùng. |
| `"always_require"` | Mỗi lần gọi phải được người dùng phê duyệt trước khi thực thi. |

Sử dụng `"always_require"` cho các công cụ có tác động phụ (ví dụ: đặt vé bay, tính phí thẻ tín dụng) để đảm bảo con người luôn tham gia vào quá trình.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Tóm tắt

Trong bài học này bạn đã học cách:

1. **Định nghĩa công cụ** sử dụng decorator `@tool` với các tham số có kiểu và docstring để làm schema cho công cụ.
2. **Kết hợp nhiều công cụ** để agent có thể gọi chúng theo tuần tự nhằm trả lời các truy vấn phức tạp.
3. **Trả về đầu ra có cấu trúc** bằng cách truyền một mô hình Pydantic làm `response_format`.
4. **Kiểm soát việc phê duyệt công cụ** với `approval_mode` để giữ con người trong vòng lặp cho các thao tác nhạy cảm.

Các mẫu này tạo thành nền tảng để xây dựng các agent đáng tin cậy, sẵn sàng cho sản xuất và có thể tương tác an toàn với các hệ thống bên ngoài.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
